# Loading

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
from scipy import sparse
from tqdm import tqdm
import glob
from sklearn.neighbors import NearestNeighbors

In [ ]:
adata = sc.read_h5ad("fetal_brain_data_ChP/mhb.h5ad")
adata

In [ ]:
for celltype in set(adata.obs['dmt_leiden_anno']):
    cell_counts = len(adata[adata.obs['dmt_leiden_anno']==celltype].obs_names)

    print(f"The celltype number of {celltype} is {cell_counts}:")
    print("--------------------------------------------------")

# h5ad enhancement

In [ ]:
from enhance_h5ad_fast import enhance_h5ad_neighbors_mem_efficient
adata.obs_names = adata.obs_names + adata.obs['slice_code'].tolist()
adata = enhance_h5ad_neighbors_mem_efficient(adata,
                       use_rep='latent_embeddings_all_single_pretrain',
                       annotation='dmt_leiden_anno',temp_path='Process_Data/specific_region/mhb_tmp',
                       num_top=50)
adata.write_h5ad("Process_Data/specific_region/enhanced_mhb.h5ad",compression='gzip')

#  mh_sc_02 h5ad enhancemnet

In [ ]:
adata.obs_names = adata.obs_names + adata.obs['slice_code'].tolist()
adata_raw = adata[adata.obs['dmt_leiden_anno']=='mh_sc_02']
adata_raw.write_h5ad("Process_Data/specific_region/mhb_tmp/raw_mh_sc_02.h5ad",compression='gzip')

In [ ]:
adata = sc.read_h5ad("Process_Data/specific_region/mhb_tmp/raw_mh_sc_02.h5ad")
adata

In [ ]:
from enhance_h5ad_fast import _row_mean_sparse_rows,_get_expression_matrix

rep = adata.obsm['latent_embeddings_all_single_pretrain']
if hasattr(rep, "toarray"):
    rep = rep.toarray()
rep = np.asarray(rep, dtype=np.float32)
X = _get_expression_matrix(adata, 'X')
n_obs, n_vars = adata.shape
enhanced_X = sparse.lil_matrix((n_obs, n_vars), dtype=np.float32)

k_to_find = min(50, n_obs)
nn = NearestNeighbors(
    n_neighbors=k_to_find,
    metric='cosine',
    n_jobs=128,
    algorithm='brute'  # <-- 强制使用蛮力算法
)
nn.fit(rep)

all_neighbor_indices = np.empty((n_obs, k_to_find), dtype=np.int32)
for start_idx in tqdm(range(0, n_obs, 12192), desc="Finding neighbors"):
    end_idx = min(start_idx + 2048, n_obs)
    query_batch = rep[start_idx:end_idx, :]
    _, batch_indices = nn.kneighbors(query_batch)
    all_neighbor_indices[start_idx:end_idx, :] = batch_indices

In [ ]:
np.savetxt('Process_Data/specific_region/mhb_tmp/all_neighbor_indices.csv', all_neighbor_indices, delimiter=',')
#all_neighbor_indices = np.loadtxt('Process_Data/specific_region/mhb_tmp/all_neighbor_indices.csv', delimiter=',')

In [ ]:
for i in tqdm(range(n_obs), desc="Aggregating neighbors"):
    neighbor_global_idx = all_neighbor_indices[i]
    
    if sparse.issparse(X):
        mean_expr = _row_mean_sparse_rows(X, neighbor_global_idx)
    else:
        if len(neighbor_global_idx) == 0:
            mean_expr = np.zeros(n_vars, dtype=np.float32)
        else:
            sub = X[neighbor_global_idx, :]
            mean_expr = np.asarray(sub, dtype=np.float32).mean(axis=0)

    enhanced_X[i, :] = mean_expr

enhanced_X_csr = enhanced_X.tocsr()
enhanced_adata = sc.AnnData(
        X=enhanced_X_csr,
        obs=adata.obs.copy(),
        var=adata.var.copy(),
        obsm=adata.obsm.copy(),
        uns=adata.uns.copy(),
    )
enhanced_adata.write_h5ad("Process_Data/specific_region/mhb_tmp/enhanced_group_mh_sc_02.h5ad",compression='gzip')

# AUCell calculation

In [ ]:
import os
import glob
import anndata
import omicverse as ov
import gc
from tqdm.auto import tqdm
from typing import Dict, List

# Prevent OpenBLAS crashes by limiting threads
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

def batch_compute_aucell_from_folder(
    input_dir: str,
    output_dir: str,
    pathway_dict: Dict[str, List[str]],
    file_prefix: str = "enhanced_group_mh", 
    num_workers: int = 10
):
    """
    Batch process h5ad files: Read -> Calculate AUCell -> Save -> Cleanup.
    """
    
    if not os.path.exists(input_dir):
        raise FileNotFoundError(f"Input directory not found: {input_dir}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    search_pattern = os.path.join(input_dir, f"{file_prefix}*.h5ad")
    found_files = glob.glob(search_pattern)
    
    if not found_files:
        print(f"Warning: No files found matching '{file_prefix}' in {input_dir}")
        return

    print(f"Found {len(found_files)} files to process.")

    for fpath in found_files:
        filename = os.path.basename(fpath)
        
        # Rename: enhanced_group_... -> auc_anndata_group_...
        if filename.startswith("enhanced_"):
            new_filename = filename.replace("enhanced_", "auc_anndata_", 1)
        else:
            new_filename = f"auc_anndata_{filename}"
            
        output_path = os.path.join(output_dir, new_filename)
        
        if os.path.exists(output_path):
            # Skip if already processed
            continue

        try:
            print(f'process {fpath}')
            adata = anndata.read_h5ad(fpath)
            
            adata_processed = ov.single.pathway_aucell_enrichment(
                adata,
                pathways_dict=pathway_dict,
                AUC_threshold=0.05,
                num_workers=num_workers
            )
            
            adata_processed.write_h5ad(output_path, compression='gzip')
            
            # Strict memory cleanup
            del adata
            del adata_processed
            gc.collect()
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            gc.collect()
            continue

    print("Batch processing complete.")

In [ ]:
import json
with open('Output/MHB/MHB/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
batch_compute_aucell_from_folder(
         input_dir='Process_Data/specific_region/mhb_tmp',
         output_dir='Process_Data/specific_region/auc_mhb_adata',
         pathway_dict=grn_dict,  
         num_workers=10)

In [ ]:
files = sorted(glob.glob(os.path.join('Process_Data/specific_region/auc_mhb_adata', "*.h5ad")))
adatas = []
for f in tqdm(files):
    print(f'process {f}')
    auc_adata = sc.read_h5ad(f)
    obs_adata = sc.read_h5ad(os.path.join('Process_Data/specific_region/mhb_tmp','enhanced_group_'+'_'.join(f.split('/')[-1].split('_')[3:6])))
    auc_adata.obs = obs_adata[auc_adata.obs_names,:].obs
    auc_adata.obsm = obs_adata[auc_adata.obs_names,:].obsm
    auc_adata.write_h5ad(f, compression='gzip')
    #adatas.append(auc_adata)
#combined_adata = ad.concat(adatas, axis=0, join='outer', merge='same')

In [ ]:
files = sorted(glob.glob(os.path.join('Process_Data/specific_region/auc_mhb_adata', "*.h5ad")))
adatas = []
for f in tqdm(files):
    print(f'process {f}')
    auc_adata = sc.read_h5ad(f)
    adatas.append(auc_adata)
combined_adata = ad.concat(adatas, axis=0, join='outer', merge='same')
combined_adata.write_h5ad("Process_Data/specific_region/auc_mhb.h5ad",
                            compression="gzip")